# Tagged corpus analysis execution summary

This notebook is a local reproducibility summary based on saved repository artifacts.
It is not labeled as a Colab execution artifact because it is executed with nbclient in the local environment.

In [1]:
import importlib.util
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
if importlib.util.find_spec("torch"):
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
else:
    print("Torch: not installed")
    print("CUDA available: False")
    print("GPU: none")
try:
    print("Git commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
except Exception as exc:
    print("Git commit: unavailable", type(exc).__name__, exc)


Python: 3.11.1
Platform: Windows-10-10.0.26200-SP0


Torch: 2.0.1+cpu
CUDA available: False
GPU: none
Git commit: 054f093e8a456cff34063272aabf0f8522a3380e


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

dataset_path = Path("data/processed/anekdots_tagged.csv")
clustered_path = Path("data/processed/anekdots_tagged_clustered.csv")
embedding_path = Path("data/embeddings/tagged_bge_m3.npy")
pca_path = Path("data/embeddings/tagged_pca128.npy")
final_metrics_path = Path("outputs/tables/final_metrics_summary.csv")
before_after_path = Path("outputs/tables/metrics_before_after.csv")
selection_path = Path("outputs/tables/final_clustering_selection.csv")

dataset = pd.read_csv(dataset_path)
clustered = pd.read_csv(clustered_path)
embeddings = np.load(embedding_path)
pca = np.load(pca_path)
final_metrics = pd.read_csv(final_metrics_path)
before_after = pd.read_csv(before_after_path)
selection = pd.read_csv(selection_path)

print("Read:", dataset_path, len(dataset))
print("Read:", clustered_path, len(clustered))
print("Embeddings shape:", embeddings.shape)
print("PCA shape:", pca.shape)
print("Final metrics rows:", len(final_metrics))
print("Before/after rows:", len(before_after))
print("Selection rows:", len(selection))


Read: data\processed\anekdots_tagged.csv 5509
Read: data\processed\anekdots_tagged_clustered.csv 5509
Embeddings shape: (5509, 1024)
PCA shape: (5509, 128)
Final metrics rows: 15
Before/after rows: 7
Selection rows: 1


In [3]:
selected = selection.iloc[0]
cluster_count = int(clustered["cluster_final"].nunique())
largest_cluster_share = float(clustered["cluster_final"].value_counts(normalize=True).max())

print("Final method:", selected["method"])
print("Final feature set:", selected["feature_set"])
print("Final params:", selected["params"])
print("Final cluster count:", cluster_count)
print("Largest cluster share:", round(largest_cluster_share, 6))


Final method: leiden
Final feature set: hybrid_dense_lexical_dw0.75_lw0.25
Final params: k=75;resolution=2.0;seed=7
Final cluster count: 20
Largest cluster share: 0.129062


In [4]:
final_all = final_metrics[(final_metrics["model"] == "final") & (final_metrics["subset"] == "all")].iloc[0]
single = final_metrics[(final_metrics["model"] == "final") & (final_metrics["subset"] == "single_clear_label")].iloc[0]

print("Final ARI:", round(float(final_all["ari"]), 6))
print("Final AMI:", round(float(final_all["ami"]), 6))
print("Final V-measure:", round(float(final_all["v_measure"]), 6))
print("Final pairwise precision:", round(float(final_all["pairwise_precision"]), 6))
print("Final pairwise recall:", round(float(final_all["pairwise_recall"]), 6))
print("Final pairwise F1:", round(float(final_all["pairwise_f1"]), 6))
print("Single-clear-label V-measure:", round(float(single["v_measure"]), 6))
print("Single-clear-label pairwise F1:", round(float(single["pairwise_f1"]), 6))
print(before_after[["metric", "old_leiden", "final", "delta"]].to_string(index=False))


Final ARI: 0.27684
Final AMI: 0.377156
Final V-measure: 0.387097
Final pairwise precision: 0.351852
Final pairwise recall: 0.326731
Final pairwise F1: 0.338826
Single-clear-label V-measure: 0.419784
Single-clear-label pairwise F1: 0.354086
                   metric  old_leiden    final     delta
      ari_excluding_other    0.196673 0.276840  0.080167
                      ami    0.303855 0.377156  0.073301
v_measure_excluding_other    0.310895 0.387097  0.076203
          pairwise_f1_all    0.284616 0.338826  0.054210
       pairwise_precision    0.229705 0.351852  0.122147
          pairwise_recall    0.374027 0.326731 -0.047296
    largest_cluster_share    0.204574 0.129062 -0.075513


In [5]:
generated_report_artifacts = [
    "outputs/tables/final_clustering_selection.csv",
    "outputs/tables/final_metrics_summary.csv",
    "outputs/tables/final_internal_cluster_metrics.csv",
    "outputs/tables/metrics_before_after.csv",
    "outputs/tables/cluster_ctfidf_terms.csv",
    "outputs/tables/cluster_interpretation_cards.csv",
    "outputs/tables/cluster_report_ready_summary.csv",
    "outputs/tables/macro_tag_mapping_audit.csv",
    "outputs/report_notes/10_final_clustering_selection.md",
    "outputs/report_notes/11_cluster_interpretation.md",
    "outputs/report_notes/12_macro_tag_mapping_audit.md",
    "outputs/figures/umap2d_final.html",
    "outputs/figures/umap3d_final.html",
    "outputs/figures/umap2d_final.png",
    "outputs/figures/umap3d_final.png",
]
for artifact in generated_report_artifacts:
    path = Path(artifact)
    print(artifact, "exists=", path.exists(), "bytes=", path.stat().st_size if path.exists() else 0)


outputs/tables/final_clustering_selection.csv exists= True bytes= 954
outputs/tables/final_metrics_summary.csv exists= True bytes= 5857
outputs/tables/final_internal_cluster_metrics.csv exists= True bytes= 420
outputs/tables/metrics_before_after.csv exists= True bytes= 573
outputs/tables/cluster_ctfidf_terms.csv exists= True bytes= 16291
outputs/tables/cluster_interpretation_cards.csv exists= True bytes= 14772
outputs/tables/cluster_report_ready_summary.csv exists= True bytes= 17912
outputs/tables/macro_tag_mapping_audit.csv exists= True bytes= 26808
outputs/report_notes/10_final_clustering_selection.md exists= True bytes= 14190
outputs/report_notes/11_cluster_interpretation.md exists= True bytes= 21034
outputs/report_notes/12_macro_tag_mapping_audit.md exists= True bytes= 5184
outputs/figures/umap2d_final.html exists= True bytes= 4679540
outputs/figures/umap3d_final.html exists= True bytes= 4753251
outputs/figures/umap2d_final.png exists= True bytes= 582432
outputs/figures/umap3d_fina